# BASA JAWA Edge — Fine tuning DistilBERT opsional
Jalankan setelah notebook 01. Eksperimen ini menguji model dari jalur penelitian terdahulu pada pembagian data yang sama. Gunakan GPU Colab dan simpan hasil sendiri. Tidak ada hasil fine-tuning yang sudah diklaim dalam paket.

Backbone: `w11wo/javanese-distilbert-small-imdb`, sebuah masked language model pada korpus ulasan Jawa. Tambahkan kepala klasifikasi empat label dan fine-tune hanya pada train. Hindari checkpoint honorifik yang sudah dilatih pada Unggah-Ungguh untuk pembandingan test yang bersih.

PWA saat ini mendukung LR/SVM/NB. Checkpoint DistilBERT menjadi pembanding penelitian, bukan aset browser pada runtime ini.


## 1 Ambil proyek dengan cara yang sama


In [ ]:
import os,sys,subprocess,re,json
from pathlib import Path
for name in ['OPENBLAS_NUM_THREADS','OMP_NUM_THREADS','MKL_NUM_THREADS']:
    os.environ[name]='1'
# Isi alamat repositori publik Anda setelah mengunggah paket awal ke GitHub.
REPO_URL = "https://github.com/USERNAME/basa-jawa-edge.git"
REPO_REF = "main"   # Untuk mengulang persis, ganti dengan commit SHA yang telah dicatat.
MODE = "github"     # Pilihan lain: "upload_zip" bila repositori belum dibuat.
if os.environ.get("BASA_REPO_ROOT"):
    ROOT = Path(os.environ["BASA_REPO_ROOT"]).resolve()  # pemeriksaan lokal oleh pengembang
elif MODE == "github":
    if not re.fullmatch(r"https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?",REPO_URL) or "USERNAME" in REPO_URL:
        raise ValueError("Ganti REPO_URL dengan repositori GitHub Anda, lalu jalankan sel ini lagi.")
    ROOT = Path("/content/basa-jawa-edge")
    if not ROOT.exists():subprocess.run(["git","clone",REPO_URL,str(ROOT)],check=True)
    subprocess.run(["git","-C",str(ROOT),"fetch","origin"],check=True)
    resolved_ref='origin/main' if REPO_REF=='main' else REPO_REF
    # Detached checkout mengunci sumber pada commit yang benar-benar diambil.
    # Git akan berhenti bila ada perubahan lokal yang bertabrakan; tidak ada reset otomatis.
    subprocess.run(["git","-C",str(ROOT),"checkout","--detach",resolved_ref],check=True)
else:
    from google.colab import files
    import zipfile
    uploads=files.upload()
    archive=next((name for name in uploads if name.endswith('.zip')),None)
    if archive is None:raise ValueError("Pilih ZIP paket repositori.")
    target=Path('/content/basa_upload');target.mkdir(exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            path=(target/member.filename).resolve()
            if not path.is_relative_to(target.resolve()):raise ValueError('Path ZIP tidak aman')
        z.extractall(target)
    matches=list(target.rglob('protocol/experiment.json'))
    if len(matches)!=1:raise ValueError('ZIP harus memuat satu folder proyek')
    ROOT=matches[0].parent.parent
assert (ROOT/'src/lab.py').exists(), ROOT
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print('Folder proyek:',ROOT)


## 2 Siapkan pustaka dan periksa GPU
Versi aktual akan dicatat. Jalur ini belum dijalankan pada GPU Colab dalam paket yang diberikan.


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.57.3','huggingface_hub>=0.34,<1.0','scikit-learn==1.8.0'],check=True)
import torch,transformers,numpy as np
from huggingface_hub import HfApi
from transformers import AutoTokenizer
from src import lab,deep_optional
if not torch.cuda.is_available():raise RuntimeError('Pilih Runtime → Change runtime type → GPU. Ketersediaan GPU Colab bervariasi.')
print('GPU:',torch.cuda.get_device_name(0),'Torch:',torch.__version__,'Transformers:',transformers.__version__)


## 3 Gunakan partisi yang tersimpan dari eksperimen utama
Notebook ini memerlukan hasil notebook 01 pada repositori/runtime yang sama. Jika runtime berganti, unggah kembali folder run yang disimpan. Pilih main run sendiri agar tidak memakai contoh reference_local secara tidak sengaja.


In [ ]:
MAIN_RUN_ID = 'ISI_RUN_ID_DARI_NOTEBOOK_01'
if MAIN_RUN_ID.startswith('ISI_'):raise ValueError('Isi MAIN_RUN_ID dengan ID eksperimen utama Anda.')
MAIN_RUN=ROOT/'runs'/MAIN_RUN_ID
CONFIG=json.loads((MAIN_RUN/'protocol.json').read_text())
ROWS,DIALOGUES=lab.load_snapshot(ROOT,CONFIG,MAIN_RUN)
PARTS,GROUPS=lab.partition(ROWS,CONFIG)
import csv
with (MAIN_RUN/'partitions.csv').open() as f:saved=list(csv.DictReader(f))
for name,ids in PARTS.items():assert set(map(int,ids))=={int(r['row_index']) for r in saved if r['partition']==name}
MODEL_ID='w11wo/javanese-distilbert-small-imdb'
REVISION=HfApi().model_info(MODEL_ID).sha
DEEP_RUN=MAIN_RUN/'optional_distilbert';DEEP_RUN.mkdir(exist_ok=True)
settings={'model_id':MODEL_ID,'resolved_revision':REVISION,'learning_rates':[2e-5,5e-5],'epochs_per_rate':2,'max_length':128,'batch_size':16,'seed':42,'main_run_id':MAIN_RUN_ID,'code_sha256':lab.digest(ROOT/'src/deep_optional.py'),'torch':torch.__version__,'transformers':transformers.__version__,'gpu':torch.cuda.get_device_name(0)}
lab.write_json(DEEP_RUN/'protocol_and_environment.json',settings)
TOKENIZER=AutoTokenizer.from_pretrained(MODEL_ID,revision=REVISION,trust_remote_code=False)
print(json.dumps(settings,indent=2))


## 4 Latih dua learning rate dan simpan loss setiap epoch
Loop menggunakan AdamW dan loss berbobot kelas. Fungsi fit_one mencetak langkah dan validation macro-F1. Semua keputusan memakai validasi. Bobot terbaik tiap learning rate disimpan sementara di RAM; unduh checkpoint terpilih setelah evaluasi.


In [ ]:
CANDIDATES=[]
for learning_rate in settings['learning_rates']:
    state,history=deep_optional.fit_one(MODEL_ID,REVISION,learning_rate,ROWS,PARTS,TOKENIZER,DEEP_RUN,
        epochs=settings['epochs_per_rate'],max_length=settings['max_length'],batch_size=settings['batch_size'])
    best=max(history,key=lambda r:r['validation_macro_f1'])
    CANDIDATES.append({'state':state,'best':best})
WINNER=max(CANDIDATES,key=lambda r:r['best']['validation_macro_f1'])
lab.write_json(DEEP_RUN/'selection.json',{'selected':WINNER['best'],'test_used_for_selection':False})
print('Terpilih dari validasi:',WINNER['best'])


## 5 Evaluasi test setelah tuning selesai


In [ ]:
METRICS,CHECKPOINT=deep_optional.evaluate_selected(MODEL_ID,REVISION,WINNER['state'],TOKENIZER,ROWS,PARTS,DEEP_RUN,
    max_length=settings['max_length'],batch_size=settings['batch_size'])
print(json.dumps(METRICS,indent=2));print('Checkpoint:',CHECKPOINT)


## 6 Simpan bukti dan checkpoint ke tempat yang sesuai
Simpan CSV/JSON kecil ke GitHub. Checkpoint besar simpan ke Google Drive atau tempat model yang sesuai dan catat lokasinya; jangan memasukkan bobot besar ke Git biasa. Batas GitHub untuk berkas biasa adalah 100 MiB dan unggahan browser 25 MiB.

Bandingkan macro-F1 dengan classifier ringan pada baris test yang sama. Catat GPU Colab dan versi model. Jangan membandingkan durasi GPU dengan waktu browser sebagai kondisi setara. Ekspor ONNX/transformer browser memerlukan pekerjaan dan pengujian terpisah; notebook ini tidak mengganti model PWA.

Sumber backbone: https://huggingface.co/w11wo/javanese-distilbert-small-imdb ; penelitian honorifik: https://aclanthology.org/2025.acl-long.1296/ .
